# Cancellation Model Framing

This notebook defines the machine learning target, removes leakage-prone columns, selects pre-arrival features, and creates the train/test split for cancellation prediction.

In [ ]:
# ADD IMPORTS AND LOAD DATA

import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/processed/hotel_bookings_cleaned.csv")

df.shape

(119390, 36)

In [2]:
# DEFINE TARGET

target = "is_canceled"

df[target].value_counts(normalize=True) * 100

is_canceled
0    62.958372
1    37.041628
Name: proportion, dtype: float64

In [3]:
# REMOVE LEAKAGE COLUMNS

leakage_columns = [
    "reservation_status",
    "reservation_status_date"
]

df_model = df.drop(columns=leakage_columns, errors="ignore")

df_model.shape

(119390, 34)

In [ ]:
# BASELINE FEATURES

selected_features = [
    "hotel",
    "lead_time",
    "arrival_date_month",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
    "meal",
    "country",
    "market_segment",
    "distribution_channel",
    "is_repeated_guest",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "reserved_room_type",
    "booking_changes",
    "deposit_type",
    "agent",
    "company",
    "days_in_waiting_list",
    "customer_type",
    "adr",
    "required_car_parking_spaces",
    "total_of_special_requests",
    "total_nights",
    "total_guests",
    "arrival_date_num",
    "stay_type"
]

X = df_model[selected_features]
y = df_model[target]

X.shape, y.shape

((119390, 29), (119390,))

In [ ]:
# CREATE TRAIN/TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((95512, 29), (23878, 29), (95512,), (23878,))

In [7]:
# CHECK TARGET BALANCE

print("Training target distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True) * 100)

Training target distribution:
is_canceled
0    62.958581
1    37.041419
Name: proportion, dtype: float64

Testing target distribution:
is_canceled
0    62.957534
1    37.042466
Name: proportion, dtype: float64


In [8]:
# SAVE BASELINE MODELLING DATASETS

train_data = X_train.copy()
train_data[target] = y_train

test_data = X_test.copy()
test_data[target] = y_test

train_data.to_csv("../data/processed/model_train.csv", index=False)
test_data.to_csv("../data/processed/model_test.csv", index=False)

print("Model train and test datasets saved successfully.")

Model train and test datasets saved successfully.


# Day 11 Summary

## Model Framing Completed

### Target Variable
- is_canceled

### Data Leakage Prevention
Removed:
- reservation_status
- reservation_status_date

### Baseline Features
Selected 29 booking-related features available before cancellation occurs.

### Train-Test Split
- Training Set: 95,512 rows
- Testing Set: 23,878 rows
- Split Ratio: 80:20

### Class Distribution
Training:
- Not Cancelled: 62.96%
- Cancelled: 37.04%

Testing:
- Not Cancelled: 62.96%
- Cancelled: 37.04%

### Outcome
Prepared clean train and test datasets for machine learning model development.

# Day 12 - Feature Engineering for Cancellation Model

This section creates engineered features for the cancellation prediction model and prepares encoded train/test datasets for model training.

In [9]:
import pandas as pd
import numpy as np

train_data = pd.read_csv("../data/processed/model_train.csv")
test_data = pd.read_csv("../data/processed/model_test.csv")

train_data.shape, test_data.shape

((95512, 30), (23878, 30))

In [10]:
def engineer_features(df):
    df = df.copy()

    # Lead time buckets
    df["lead_time_bucket"] = pd.cut(
        df["lead_time"],
        bins=[-1, 7, 30, 90, 180, 400, 800],
        labels=["0-7 days", "8-30 days", "31-90 days", "91-180 days", "181-400 days", "400+ days"]
    )

    # Party size
    df["party_size"] = df["adults"] + df["children"] + df["babies"]

    df["party_size_category"] = pd.cut(
        df["party_size"],
        bins=[-1, 1, 2, 4, 20],
        labels=["Solo", "Couple", "Small Group", "Large Group"]
    )

    # Has children flag
    df["has_children"] = ((df["children"] + df["babies"]) > 0).astype(int)

    # Prior cancellation flag
    df["prior_cancel_flag"] = (df["previous_cancellations"] > 0).astype(int)

    # Stay length bucket
    df["stay_length_bucket"] = pd.cut(
        df["total_nights"],
        bins=[-1, 2, 5, 10, 100],
        labels=["Short", "Medium", "Long", "Extended"]
    )

    # Season
    season_map = {
        "December": "Winter", "January": "Winter", "February": "Winter",
        "March": "Spring", "April": "Spring", "May": "Spring",
        "June": "Summer", "July": "Summer", "August": "Summer",
        "September": "Autumn", "October": "Autumn", "November": "Autumn"
    }

    df["season"] = df["arrival_date_month"].map(season_map)

    # ADR outlier treatment
    df["adr_cleaned"] = df["adr"].clip(lower=0, upper=300)

    return df

In [11]:
# APPLY FEATURE ENGINEERING

train_fe = engineer_features(train_data)
test_fe = engineer_features(test_data)

train_fe.shape, test_fe.shape

((95512, 38), (23878, 38))

In [ ]:
# CHECK NEW FEATURES

new_features = [
    "lead_time_bucket",
    "party_size",
    "party_size_category",
    "has_children",
    "prior_cancel_flag",
    "stay_length_bucket",
    "season",
    "adr_cleaned"
]

train_fe[new_features].head()

,lead_time_bucket,party_size,party_size_category,has_children,prior_cancel_flag,stay_length_bucket,season,adr_cleaned
0,8-30 days,1.0,Solo,0,0,Short,Winter,98.0
1,8-30 days,2.0,Couple,0,0,Medium,Spring,100.0
2,91-180 days,2.0,Couple,0,0,Medium,Spring,95.0
3,31-90 days,2.0,Couple,0,0,Extended,Autumn,54.0
4,8-30 days,1.0,Solo,0,0,Short,Summer,80.0


In [15]:
# ENCODE CATEGORICAL COLUMNS

target = "is_canceled"

X_train = train_fe.drop(columns=[target])
y_train = train_fe[target]

X_test = test_fe.drop(columns=[target])
y_test = test_fe[target]

categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

categorical_cols

X_train_encoded = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

X_train_encoded, X_test_encoded = X_train_encoded.align(
    X_test_encoded,
    join="left",
    axis=1,
    fill_value=0
)

X_train_encoded.shape, X_test_encoded.shape

C:\Users\Jai Shree Shyam\AppData\Local\Temp\ipykernel_15052\30475012.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()


((95512, 252), (23878, 252))

In [16]:
# SAVE ENGINEERED DATASETS

train_encoded = X_train_encoded.copy()
train_encoded[target] = y_train.values

test_encoded = X_test_encoded.copy()
test_encoded[target] = y_test.values

train_encoded.to_csv("../data/processed/model_train_engineered.csv", index=False)
test_encoded.to_csv("../data/processed/model_test_engineered.csv", index=False)

print("Engineered train and test datasets saved successfully.")

Engineered train and test datasets saved successfully.


# Day 12 Summary

## Completed

- Created lead_time_bucket
- Created party_size
- Created party_size_category
- Created has_children flag
- Created prior_cancel_flag
- Created stay_length_bucket
- Created season
- Created adr_cleaned with outlier treatment
- Encoded categorical variables
- Saved engineered train/test datasets

## Output Files

- data/processed/model_train_engineered.csv
- data/processed/model_test_engineered.csv


# Model Training
In this section, we train baseline and improved machine learning models for hotel booking cancellation prediction.

Models:
1. Logistic Regression - baseline model
2. Random Forest - improved model

Evaluation Metrics:
- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
- Confusion Matrix

In [3]:
# IMPORT LIBRARIES

import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import warnings
warnings.filterwarnings("ignore")

In [6]:
# LOAD ENGINEERED DATASETS

train_data = pd.read_csv("../data/processed/model_train_engineered.csv")
test_data = pd.read_csv("../data/processed/model_test_engineered.csv")

print("Train Shape:", train_data.shape)
print("Test Shape:", test_data.shape)

Train Shape: (95512, 253)
Test Shape: (23878, 253)


In [7]:
target = "is_canceled"

X_train = train_data.drop(columns=[target])
y_train = train_data[target]

X_test = test_data.drop(columns=[target])
y_test = test_data[target]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (95512, 252)
X_test: (23878, 252)
y_train: (95512,)
y_test: (23878,)


In [8]:
# TRAIN LOGISTIC REGRESSION

log_reg = LogisticRegression(
    solver="saga",
    max_iter=50,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

log_reg.fit(X_train, y_train)

print("Logistic Regression Training Complete")

Logistic Regression Training Complete


In [9]:
# LOGISTIC REGRESSION PREDICTIONS

y_pred_lr = log_reg.predict(X_test)
y_prob_lr = log_reg.predict_proba(X_test)[:, 1]

print("Predictions generated successfully")

Predictions generated successfully


In [10]:
# LOGISTIC REGRESSION METRICS

lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)
lr_auc = roc_auc_score(y_test, y_prob_lr)

print("Logistic Regression Results")
print("---------------------------")
print("Accuracy :", round(lr_accuracy, 4))
print("Precision:", round(lr_precision, 4))
print("Recall   :", round(lr_recall, 4))
print("F1 Score :", round(lr_f1, 4))
print("ROC AUC  :", round(lr_auc, 4))

Logistic Regression Results
---------------------------
Accuracy : 0.6495
Precision: 0.5227
Recall   : 0.6196
F1 Score : 0.567
ROC AUC  : 0.6963


In [11]:
# CONFUSION MATRIX

cm_lr = confusion_matrix(y_test, y_pred_lr)

print("Confusion Matrix:")
print(cm_lr)

Confusion Matrix:
[[10029  5004]
 [ 3365  5480]]


In [12]:
# CLASSIFICATION REPORT

print(classification_report(y_test, y_pred_lr))

              precision    recall  f1-score   support

           0       0.75      0.67      0.71     15033
           1       0.52      0.62      0.57      8845

    accuracy                           0.65     23878
   macro avg       0.64      0.64      0.64     23878
weighted avg       0.67      0.65      0.65     23878



In [14]:
# RANDOM FOREST MODEL

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)

rf_model.fit(X_train, y_train)

# print("Random Forest Training Complete")

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",10
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total 

In [15]:
# RANDOM FOREST PREDICTIONS

y_pred_rf = rf_model.predict(X_test)

y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest Predictions Complete")

Random Forest Predictions Complete


In [16]:
# RANDOM FOREST METRICS

rf_accuracy = accuracy_score(y_test, y_pred_rf)

rf_precision = precision_score(y_test, y_pred_rf)

rf_recall = recall_score(y_test, y_pred_rf)

rf_f1 = f1_score(y_test, y_pred_rf)

rf_auc = roc_auc_score(y_test, y_prob_rf)

print("Random Forest Results")
print("---------------------")

print("Accuracy :", round(rf_accuracy, 4))
print("Precision:", round(rf_precision, 4))
print("Recall   :", round(rf_recall, 4))
print("F1 Score :", round(rf_f1, 4))
print("ROC AUC  :", round(rf_auc, 4))

Random Forest Results
---------------------
Accuracy : 0.8467
Precision: 0.9121
Recall   : 0.6487
F1 Score : 0.7582
ROC AUC  : 0.9307


In [19]:
# RANDOM FOREST CONFUSION MATRIX

cm_rf = confusion_matrix(y_test, y_pred_rf)

print(cm_rf)

[[14480   553]
 [ 3107  5738]]


In [18]:
# CLASSIFICATION REPORT

print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.82      0.96      0.89     15033
           1       0.91      0.65      0.76      8845

    accuracy                           0.85     23878
   macro avg       0.87      0.81      0.82     23878
weighted avg       0.86      0.85      0.84     23878



In [20]:
comparison_df = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [lr_accuracy, rf_accuracy],
    "Precision": [lr_precision, rf_precision],
    "Recall": [lr_recall, rf_recall],
    "F1 Score": [lr_f1, rf_f1],
    "ROC AUC": [lr_auc, rf_auc]
})

comparison_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,Logistic Regression,0.649510,0.522701,0.619559,0.567024,0.696262
1,Random Forest,0.846721,0.912097,0.648728,0.758192,0.930710


In [21]:
comparison_df.to_csv(
    "../reports/day13_model_metrics.csv",
    index=False
)

print("Metrics Saved")

Metrics Saved


In [22]:
import joblib

joblib.dump(
    rf_model,
    "../models/random_forest_cancellation_model.pkl"
)

print("Model Saved Successfully")

Model Saved Successfully


In [23]:
loaded_model = joblib.load(
    "../models/random_forest_cancellation_model.pkl"
)

print(type(loaded_model))

<class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [24]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance.head(20)

,Feature,Importance
231,deposit_type_Non Refund,0.176924
168,country_PRT,0.107978
0,lead_time,0.077255
15,total_of_special_requests,0.070407
21,prior_cancel_flag,0.038134
7,previous_cancellations,0.035184
214,market_segment_Groups,0.033034
234,customer_type_Transient,0.031974
10,agent,0.030084
14,required_car_parking_spaces,0.029059


In [25]:
feature_importance.to_csv(
    "../reports/day14_feature_importance.csv",
    index=False
)

print("Feature Importance Saved")

Feature Importance Saved
